# RideBase 10 — V1.3 Final Regression

RideBase Synthetic Dataset **v1.3** (regression-calibrated, **FROZEN**) üzerinde leakage-free
advanced regression: native categorical boosting, hyperparameter optimization ve ensemble deneyleri.

**Amaç:** V1.3 regression learnability sınırını **data generator'a dokunmadan** ölçmek. Bu bir
**MODEL ENGINEERING** deneyidir; DATA ENGINEERING değildir. Production validation hâlâ `BLOCKED`.

### Ne yapıyoruz / neden
V1.3 generator calibration ile DAYS TEST R² ~0.64'e çıktı (v1.2 ~0.42). Soru: *daha güçlü
preprocessing + native-categorical modeller + boosting + tuning + ensemble* ile performans nereye
kadar taşınır? Dataset sabit; sadece modelleme değişir.

### Kavramlar (ilk kullanımda)
| kavram | anlamı |
|---|---|
| **one-hot encoding** | kategoriyi 0/1 sütunlara açmak (sklearn ağaçları için) |
| **native categorical** | kategoriyi ham bırakıp CatBoost/LightGBM'in kendi işlemesi |
| **boosting** | zayıf ağaçları sırayla ekleyip hatayı düşürmek (HGB/XGB/LGBM/CatBoost) |
| **early stopping** | validation hatası artınca ağaç eklemeyi durdurmak |
| **hyperparameter tuning** | model ayarlarını (derinlik, learning rate…) aramak |
| **temporal validation** | zamana saygılı fold: geçmişle eğit, gelecekte doğrula |
| **ensemble / weighted blending** | birden çok modelin tahminini (ağırlıklı) ortalamak |
| **model diversity** | modellerin hata profillerinin farklı olması (blend'e fayda) |
| **diminishing returns** | ek eforun getirdiği kazancın giderek azalması |
| **R² vs MAE** | R² açıklanan varyans oranı; MAE ortalama mutlak hata (gün/km) |

### V1.2 → V1.3 farkı
V1.2'de hedef fazla gürültülüydü (DAYS R² tavanı ~0.44). V1.3 generator, next-service
interval'ını **snapshot'ta gözlemlenebilir** sürücülerden (recent usage, historical interval,
adherence, service history, risk) üretecek şekilde kalibre edildi; latent davranış ML feature
olarak export edilmiyor. Split, target ve noise **değişmedi**.

### Bütçe
`RB_FAST=1` ortam değişkeni yalnız hızlı smoke-test içindir. Normal koşuda RandomizedSearch
(n_iter=12, 3 temporal fold), CatBoost/LightGBM/XGBoost manuel grid + early stopping ve
(kurulu ise) AutoGluon target başına ~900 sn çalışır — toplam **Run All ~40–70 dk** sürebilir.


In [13]:
from pathlib import Path
from collections import OrderedDict
import json, os, platform, warnings

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn import __version__ as sklearn_version
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (ExtraTreesRegressor, GradientBoostingRegressor,
                              HistGradientBoostingRegressor, RandomForestRegressor)
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, median_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from xgboost import XGBRegressor
except Exception:
    XGBRegressor = None
try:
    from lightgbm import LGBMRegressor
except Exception:
    LGBMRegressor = None
try:
    from catboost import CatBoostRegressor, Pool as CatPool
except Exception:
    CatBoostRegressor = None; CatPool = None
try:
    from autogluon.tabular import TabularPredictor
except Exception:
    TabularPredictor = None

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 200)

SEED = 42
np.random.seed(SEED)
FAST = os.environ.get("RB_FAST") == "1"          # yalnız hızlı yerel smoke-test için
N_ITER = 2 if FAST else 12                        # RandomizedSearch bütçesi
INNER_FOLDS = 2 if FAST else 3                    # TRAIN içi temporal fold
BOOST_GRID = 2 if FAST else 6                     # CatBoost/LGBM/XGB manuel config sayısı
AUTOML_BUDGET = 20 if FAST else 900               # AutoGluon target başına saniye
DATASET_VERSION = "1.3.0"

def find_project_root():
    here = Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c / "notebooks").is_dir() and (c / "models").is_dir():
            return c
    raise FileNotFoundError("ridebase-ml proje koku bulunamadi")

ROOT = find_project_root()
DATASET_ROOT = ROOT.parent / "ridebase_v1_3"
SOURCE = DATASET_ROOT / "source_tables"
DERIVED = DATASET_ROOT / "derived_outputs"
MODELS = ROOT / "models"; OUTPUTS = ROOT / "outputs"; REPORTS = ROOT / "reports"
TABLES = REPORTS / "tables"; FIGURES = REPORTS / "figures" / "v1_3_final_regression"
AUTOML_DIR = MODELS / "v1_3_automl"
for d in [MODELS, OUTPUTS, TABLES, FIGURES, AUTOML_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DAYS_TOL = [15, 30, 45, 60, 90]
KM_TOL = [500, 1000, 1500, 2000, 5000]

def regression_metrics(y, pred, target):
    y = np.asarray(y, float); pred = np.asarray(pred, float); ae = np.abs(pred - y)
    out = {"mae": mean_absolute_error(y, pred), "median_ae": median_absolute_error(y, pred),
           "rmse": mean_squared_error(y, pred) ** 0.5, "r2": r2_score(y, pred),
           "bias": float(np.mean(pred - y)), "p90_ae": float(np.quantile(ae, .90))}
    for x in (DAYS_TOL if target == "DAYS" else KM_TOL):
        out[f"within_{x}"] = float(np.mean(ae <= x))
    return out

def savefig(name):
    plt.tight_layout(); plt.savefig(FIGURES / name, dpi=150, bbox_inches="tight"); plt.close()

plt.style.use("seaborn-v0_8-whitegrid")
print("SETUP OK | FAST=", FAST, "| xgb", XGBRegressor is not None,
      "lgbm", LGBMRegressor is not None, "catboost", CatBoostRegressor is not None,
      "autogluon", TabularPredictor is not None)

SETUP OK | FAST= False | xgb True lgbm True catboost True autogluon True


## 6–7 — Dataset & split guard

`dataset_version == generator_version == 1.3.0` **ve** `regression_calibration_experiment == true`
zorunlu. Authoritative `split_manifest` kullanılır (random split yok). TEST; tuning / feature
seçimi / ensemble weight / early stopping / model family seçimi için **kullanılmaz** — yalnız
final aşamada açılır.

**Censoring contract:** hâlâ V1 regression → yalnız *observed* target (TRAIN 25 442, VAL 1 429,
TEST 1 282). Censored snapshot regression eğitim hedefi değildir; NULL target'a 0/-1/median
yazılmaz.


In [14]:
with open(DERIVED / "dataset_metadata.json", encoding="utf-8") as f:
    metadata = json.load(f)
info = metadata["dataset"]
if info.get("dataset_version") != DATASET_VERSION or info.get("generator_version") != DATASET_VERSION:
    raise RuntimeError("Yalniz RideBase Synthetic Dataset v1.3.0 kullanilabilir")
if metadata.get("regression_calibration_experiment") is not True:
    raise RuntimeError("regression_calibration_experiment flag beklenen degil")
print("REGRESSION_CALIBRATION_CONTEXT:", metadata["regression_calibration_description"][:160], "...")
print("NOISE_CALIBRATION:", metadata["noise_calibration"])

snapshots = pd.read_parquet(DERIVED / "ml_maintenance_snapshots.parquet")
targets = pd.read_parquet(DERIVED / "ml_next_service_targets.parquet")
split_manifest = pd.read_csv(DERIVED / "split_manifest.csv", encoding="utf-8-sig", low_memory=False)

EXPECTED_SPLIT = {"TRAIN": 32203, "VALIDATION": 4845, "TEST": 4470}
sm = split_manifest.set_index("snapshot_id")
if sm.primary_time_split.value_counts().to_dict() != EXPECTED_SPLIT:
    raise RuntimeError(f"Split guard failed: {sm.primary_time_split.value_counts().to_dict()}")
if len(snapshots) != 41518 or snapshots.snapshot_id.duplicated().any():
    raise RuntimeError("Snapshot guard failed")

tsel = targets[["snapshot_id", "target_event_observed", "is_right_censored", "days_to_next_service",
                "km_to_next_service", "target_km_valid", "next_service_type_code", "next_service_is_breakdown"]]
df = (snapshots.merge(tsel, on="snapshot_id", how="left", validate="one_to_one")
               .merge(split_manifest[["snapshot_id", "primary_time_split",
                                      "next_service_regression_eligible_primary",
                                      "primary_label_cutoff_at"]],
                      on="snapshot_id", how="left", validate="one_to_one"))
df["split"] = df["primary_time_split"].astype(str)
df["snapshot_at"] = pd.to_datetime(df["snapshot_at"])
df = df.set_index("snapshot_id", drop=False)

elig = df.next_service_regression_eligible_primary.astype(bool)
obs_mask = (elig & df.days_to_next_service.notna() & np.isfinite(df.days_to_next_service)
            & (df.days_to_next_service >= 0))
km_mask = (obs_mask & df.km_to_next_service.notna() & np.isfinite(df.km_to_next_service)
           & (df.km_to_next_service >= 0) & (df.target_km_valid == 1))
obs_counts = df.loc[obs_mask].groupby("split").size().to_dict()
km_counts = df.loc[km_mask].groupby("split").size().to_dict()
EXPECTED_OBS = {"TRAIN": 25442, "VALIDATION": 1429, "TEST": 1282}
if obs_counts != EXPECTED_OBS:
    raise RuntimeError(f"Observed regression contract mismatch: {obs_counts}")
print("DATASET_GUARD=PASS", info["dataset_version"], "| split", EXPECTED_SPLIT)
print("OBSERVED_DAYS=", obs_counts, "| OBSERVED_KM=", km_counts)

REGRESSION_CALIBRATION_CONTEXT: This release is a synthetic generator calibration experiment intended to test the learnability of next-service timing/mileage from snapshot-available features.  ...
NOISE_CALIBRATION: {'main_regular_log_scale': 0.14, 'main_irregular_log_scale': 0.19, 'bounded_factor': [0.67, 1.43], 'selection_reason': 'reasonable realism, not score maximizing', 'sensitivity_table': 'reports/tables/regression_noise_sensitivity.csv'}
DATASET_GUARD=PASS 1.3.0 | split {'TRAIN': 32203, 'VALIDATION': 4845, 'TEST': 4470}
OBSERVED_DAYS= {'TEST': 1282, 'TRAIN': 25442, 'VALIDATION': 1429} | OBSERVED_KM= {'TEST': 1282, 'TRAIN': 25442, 'VALIDATION': 1429}


## 8–9 — Feature inventory & leakage audit

**Ne yapıyoruz?** v1.3 snapshot şemasını yeniden grupluyoruz (BASE, POLICY, RECENT_USAGE,
HISTORICAL_INTERVAL, MAINTENANCE_HISTORY) ve her sütun için leakage tablosu üretiyoruz.

**Leakage riski?** `future_derived` / `target_derived` / `identifier` / `synthetic_only`
(metadata'daki `SYNTHETIC_ONLY_EXCLUDED_FROM_FINAL_ML`: `behavior_label, noise_z,
deterministic_days, deterministic_km, failure_hazard_score, chronic_risk_tier`) → `USED=false`.
Herhangi biri USED çıkarsa notebook durur. Kaydedilir: `v1_3_final_feature_leakage_audit.csv`.


In [15]:
ID_COLS = ["snapshot_id", "source_service_id", "motorcycle_id", "customer_id", "workshop_id", "model_id", "model_name"]
META_COLS = ["snapshot_at", "snapshot_date", "feature_version", "data_origin", "generator_version",
             "random_seed", "scenario_id"]
SYNTHETIC_ONLY = list(metadata["feature_availability"]["SYNTHETIC_ONLY_EXCLUDED_FROM_FINAL_ML"])
TARGET_COLS = list(tsel.columns) + ["split", "primary_time_split", "next_service_regression_eligible_primary",
                                    "primary_label_cutoff_at"]

GROUPS = OrderedDict()
GROUPS["POLICY"] = ["policy_group", "policy_ready", "policy_interval_days", "policy_interval_km",
                    "current_policy_due_task_count", "total_policy_due_task_count",
                    "maintenance_overdue_days_pre_service", "maintenance_overdue_km_pre_service"]
GROUPS["RECENT_USAGE"] = ["recent_30d_km", "recent_60d_km", "recent_90d_km", "recent_180d_km",
                          "recent_km_per_day", "long_term_km_per_day", "recent_vs_long_term_usage_ratio"]
GROUPS["HISTORICAL_INTERVAL"] = ["previous_service_count", "previous_interval_days", "previous_interval_km",
                                 "historical_interval_days_median", "historical_interval_days_std",
                                 "historical_interval_km_median", "historical_interval_km_std",
                                 "avg_service_interval_days", "avg_service_interval_km",
                                 "rolling3_interval_days", "rolling3_interval_km",
                                 "days_since_previous_service", "km_since_previous_service",
                                 "avg_km_per_day_since_previous_service", "service_sequence"]
GROUPS["MAINTENANCE_HISTORY"] = [
    "historical_policy_delay_median_days", "historical_policy_delay_mean_days", "historical_on_time_rate",
    "previous_policy_delay_days", "periodic_service_count", "repair_service_count", "breakdown_service_count",
    "warranty_service_count", "appointment_service_count", "walkin_service_count", "services_last_90d",
    "services_last_365d", "cumulative_service_spend", "avg_service_spend", "total_task_count",
    "total_completed_task_count", "total_declined_task_count", "total_fault_task_count",
    "total_inspection_finding_task_count", "total_replace_task_count",
    "days_since_engine_task", "km_since_engine_task", "days_since_brakes_task", "km_since_brakes_task",
    "days_since_final_drive_task", "km_since_final_drive_task", "days_since_tires_wheels_task",
    "km_since_tires_wheels_task", "days_since_electrical_task", "km_since_electrical_task",
    "days_since_transmission_task", "km_since_transmission_task", "days_since_cooling_task",
    "km_since_cooling_task", "days_since_intake_task", "km_since_intake_task",
    "previous_failure_count", "service_odometer_regression_count_to_date"]

NON_FEATURE = set(ID_COLS + META_COLS + SYNTHETIC_ONLY + TARGET_COLS)
grouped_feats = {c for v in GROUPS.values() for c in v}
GROUPS["BASE"] = [c for c in snapshots.columns if c not in NON_FEATURE and c not in grouped_feats]
GROUPS.move_to_end("BASE", last=False)

ALL_FEATURES = [c for g in GROUPS.values() for c in g]
CAT_FEATURES = [c for c in ALL_FEATURES if df[c].dtype == "object"]
BOOL_FEATURES = [c for c in ALL_FEATURES if df[c].dtype == "bool"]
NUM_FEATURES = [c for c in ALL_FEATURES if c not in CAT_FEATURES]
for c in BOOL_FEATURES:
    df[c] = df[c].astype("float64")

inv_rows = []
for grp, cols in GROUPS.items():
    for c in cols:
        inv_rows.append({"feature_group": grp, "feature": c, "dtype": str(df[c].dtype),
                         "kind": "categorical" if c in CAT_FEATURES else "numeric",
                         "null_rate_train": float(df.loc[df.split == "TRAIN", c].isna().mean())})
feature_inventory = pd.DataFrame(inv_rows)
feature_inventory.to_csv(TABLES / "v1_3_feature_inventory.csv", index=False, encoding="utf-8-sig")

leak_rows = []
for c in snapshots.columns:
    is_feat = c in ALL_FEATURES
    future = c in ("days_to_next_service", "km_to_next_service") or c.startswith("next_service")
    leak_rows.append({
        "feature_name": c, "source": "ml_maintenance_snapshots",
        "available_at_snapshot": c not in TARGET_COLS,
        "future_derived": future, "target_derived": c in tsel.columns,
        "identifier": c in ID_COLS, "synthetic_only": c in SYNTHETIC_ONLY,
        "production_derivable": c in metadata["feature_availability"]["PRODUCTION_DERIVABLE"],
        "used": is_feat})
leakage_audit = pd.DataFrame(leak_rows)
leakage_audit.to_csv(TABLES / "v1_3_final_feature_leakage_audit.csv", index=False, encoding="utf-8-sig")
bad = leakage_audit[(leakage_audit.used) & (leakage_audit.future_derived | leakage_audit.target_derived
                                            | leakage_audit.identifier | leakage_audit.synthetic_only)]
if len(bad):
    raise RuntimeError(f"Leakage audit failed for USED features: {bad.feature_name.tolist()}")
print(f"FEATURES total={len(ALL_FEATURES)} numeric={len(NUM_FEATURES)} categorical={len(CAT_FEATURES)}")
print("GROUP SIZES:", {g: len(c) for g, c in GROUPS.items()})
print("SYNTHETIC_ONLY excluded:", SYNTHETIC_ONLY)

FEATURES total=146 numeric=122 categorical=24
GROUP SIZES: {'BASE': 78, 'POLICY': 8, 'RECENT_USAGE': 7, 'HISTORICAL_INTERVAL': 15, 'MAINTENANCE_HISTORY': 38}
SYNTHETIC_ONLY excluded: ['behavior_label', 'noise_z', 'deterministic_days', 'deterministic_km', 'failure_hazard_score', 'chronic_risk_tier']


## 10–11 — İki representation + V1.3 preprocessor

**A) ENCODED**: numeric median-impute + missing-indicator + scale; categorical UNKNOWN-impute +
`OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=25)`. **Fit yalnız TRAIN**;
VAL/TEST sadece transform. Kaydedilir: `models/v1_3_final_preprocessor.joblib`.

**B) NATIVE**: kategorik kolonlar ham `category` dtype — CatBoost / LightGBM kendi işler.
Ham ID'ler (`snapshot_id, motorcycle_id, customer_id, workshop_id, model_id`) **feature değildir.**

V1.2 preprocessor'ı kör kullanmıyoruz; v1.3 şemasından yeni ColumnTransformer kuruluyor.


In [16]:
FEATURE_SETS = OrderedDict()
order = ["BASE", "POLICY", "RECENT_USAGE", "HISTORICAL_INTERVAL", "MAINTENANCE_HISTORY"]
cum = []
labels = ["SET_A_BASE", "SET_B_POLICY", "SET_C_RECENT_USAGE", "SET_D_HISTORICAL_INTERVAL", "SET_E_MAINT_HISTORY"]
for lab, g in zip(labels, order):
    cum = cum + GROUPS[g]
    FEATURE_SETS[lab] = list(cum)
FEATURE_SETS["SET_F_FULL"] = list(ALL_FEATURES)

is_train = (df.split == "TRAIN").to_numpy()
is_val = (df.split == "VALIDATION").to_numpy()
is_test = (df.split == "TEST").to_numpy()

def build_encoder(feat_cols):
    num = [c for c in feat_cols if c in NUM_FEATURES]
    cat = [c for c in feat_cols if c in CAT_FEATURES]
    enc = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
                          ("sc", StandardScaler())]), num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="UNKNOWN")),
                          ("oh", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=25,
                                              sparse_output=False))]), cat),
    ], remainder="drop", verbose_feature_names_out=True)
    return enc

full_encoder = build_encoder(FEATURE_SETS["SET_F_FULL"])
full_encoder.fit(df.loc[is_train, FEATURE_SETS["SET_F_FULL"]])
Xenc_full = {"TRAIN": full_encoder.transform(df.loc[is_train, FEATURE_SETS["SET_F_FULL"]]).astype("float32"),
             "VALIDATION": full_encoder.transform(df.loc[is_val, FEATURE_SETS["SET_F_FULL"]]).astype("float32"),
             "TEST": full_encoder.transform(df.loc[is_test, FEATURE_SETS["SET_F_FULL"]]).astype("float32")}
enc_feature_names = list(full_encoder.get_feature_names_out())
if not all(np.isfinite(v).all() for v in Xenc_full.values()):
    raise RuntimeError("Encoded transform NaN/inf QA failed")
joblib.dump({"encoder": full_encoder, "feature_set": FEATURE_SETS["SET_F_FULL"],
             "encoded_names": enc_feature_names, "dataset_version": DATASET_VERSION,
             "fit_scope": "TRAIN only"}, MODELS / "v1_3_final_preprocessor.joblib")

def native_frame(split):
    m = {"TRAIN": is_train, "VALIDATION": is_val, "TEST": is_test}[split]
    fr = df.loc[m, ALL_FEATURES].copy()
    for c in CAT_FEATURES:
        fr[c] = fr[c].astype("string").fillna("UNKNOWN").astype("category")
    return fr

y = {t: {s: df.loc[{"TRAIN": is_train, "VALIDATION": is_val, "TEST": is_test}[s] & obs_mask.to_numpy(),
                    "days_to_next_service" if t == "DAYS" else "km_to_next_service"].to_numpy(float)
         for s in ["TRAIN", "VALIDATION", "TEST"]}
     for t in ["DAYS", "KM"]}
obs_pos = {s: np.flatnonzero({"TRAIN": is_train, "VALIDATION": is_val, "TEST": is_test}[s] & obs_mask.to_numpy())
           for s in ["TRAIN", "VALIDATION", "TEST"]}
# encoded matrices restricted to OBSERVED rows, per split (row order preserved)
split_all_pos = {s: np.flatnonzero({"TRAIN": is_train, "VALIDATION": is_val, "TEST": is_test}[s]) for s in ["TRAIN", "VALIDATION", "TEST"]}
def enc_obs(split):
    keep = obs_mask.to_numpy()[split_all_pos[split]]
    return Xenc_full[split][keep]
Xo = {s: enc_obs(s) for s in ["TRAIN", "VALIDATION", "TEST"]}
nat = {s: native_frame(s) for s in ["TRAIN", "VALIDATION", "TEST"]}
nat_obs = {s: nat[s].loc[obs_mask.reindex(nat[s].index).to_numpy()] for s in ["TRAIN", "VALIDATION", "TEST"]}
train_time_order = np.argsort(df.iloc[obs_pos["TRAIN"]].snapshot_at.to_numpy())
print("Xo TRAIN", Xo["TRAIN"].shape, "| encoded dims", len(enc_feature_names))

Xo TRAIN (25442, 281) | encoded dims 281


## 12 — Baseline reproduction

V1.3 calibration raporundaki SAME-MODEL baseline'ı (`HistGradientBoostingRegressor` default)
yeniden üretiyoruz. Beklenen ≈ DAYS R² 0.6398 / MAE 22.78, KM R² 0.4134 / MAE 969.
Sapma toleransı: |ΔR²| < 0.03 ve |ΔMAE| < %6 → `OK`; aksi halde `DEVIATION` (durup nedenini raporla).
Bu koşunun V1.3 Basic referansı buradan alınır.


In [17]:
BASE_HGB = dict(random_state=SEED)
baseline_rows = []
baseline_models = {}
for t in ["DAYS", "KM"]:
    mdl = HistGradientBoostingRegressor(**BASE_HGB)
    mdl.fit(Xo["TRAIN"], y[t]["TRAIN"])
    baseline_models[t] = mdl
    for s in ["TRAIN", "VALIDATION", "TEST"]:
        pred = np.clip(mdl.predict(Xo[s]), 0, None)
        met = regression_metrics(y[t][s], pred, t)
        baseline_rows.append({"target": t, "split": s, "model": "HistGradientBoostingRegressor(default)", **met})
baseline_repro = pd.DataFrame(baseline_rows)
baseline_repro.to_csv(TABLES / "v1_3_baseline_reproduction.csv", index=False, encoding="utf-8-sig")
def brepro(t, s, m):
    return float(baseline_repro[(baseline_repro.target == t) & (baseline_repro.split == s)].iloc[0][m])
REPORTED = {"DAYS": {"r2": 0.6398, "mae": 22.7761}, "KM": {"r2": 0.4134, "mae": 969.41}}
repro_flags = {}
for t in ["DAYS", "KM"]:
    dr2 = abs(brepro(t, "TEST", "r2") - REPORTED[t]["r2"])
    dmae = abs(brepro(t, "TEST", "mae") - REPORTED[t]["mae"]) / REPORTED[t]["mae"]
    repro_flags[t] = "OK" if (dr2 < 0.03 and dmae < 0.06) else "DEVIATION"
    print(f"{t} baseline TEST R2 {brepro(t,'TEST','r2'):.4f} (rapor {REPORTED[t]['r2']}) | "
          f"MAE {brepro(t,'TEST','mae'):.3f} (rapor {REPORTED[t]['mae']}) -> {repro_flags[t]}")
V13_BASIC = {t: {"r2": brepro(t, "TEST", "r2"), "mae": brepro(t, "TEST", "mae"),
                 "median_ae": brepro(t, "TEST", "median_ae"), "p90_ae": brepro(t, "TEST", "p90_ae"),
                 "within_a": brepro(t, "TEST", f"within_{(DAYS_TOL if t=='DAYS' else KM_TOL)[1]}"),
                 "within_b": brepro(t, "TEST", f"within_{(DAYS_TOL if t=='DAYS' else KM_TOL)[3]}")}
             for t in ["DAYS", "KM"]}

DAYS baseline TEST R2 0.6471 (rapor 0.6398) | MAE 22.624 (rapor 22.7761) -> OK
KM baseline TEST R2 0.4250 (rapor 0.4134) | MAE 948.186 (rapor 969.41) -> OK


## 13–14 — Feature-set ablation

Aynı default HGB ile kümülatif feature setleri (SET_A…SET_F) validation'da karşılaştırılır.
Amaç: v1.3 performans artışını **hangi signal grubunun** taşıdığını görmek (recent usage,
historical interval, adherence…). Kaydedilir: `v1_3_feature_ablation.csv`.


In [18]:
def enc_for_set(setname, split):
    cols = FEATURE_SETS[setname]
    e = build_encoder(cols)
    e.fit(df.loc[is_train, cols])
    keep = obs_mask.to_numpy()[split_all_pos[split]]
    return e.transform(df.loc[{"TRAIN": is_train, "VALIDATION": is_val, "TEST": is_test}[split], cols]).astype("float32")[keep], e

abl_rows = []
prev_mae = {"DAYS": None, "KM": None}
for setname in FEATURE_SETS:
    Xtr, e = enc_for_set(setname, "TRAIN")
    Xva, _ = enc_for_set(setname, "VALIDATION")
    row = {"feature_set": setname, "feature_count": Xtr.shape[1]}
    for t in ["DAYS", "KM"]:
        mdl = HistGradientBoostingRegressor(random_state=SEED)
        mdl.fit(Xtr, y[t]["TRAIN"])
        met = regression_metrics(y[t]["VALIDATION"], np.clip(mdl.predict(Xva), 0, None), t)
        row[f"{t.lower()}_val_mae"] = met["mae"]; row[f"{t.lower()}_val_r2"] = met["r2"]
    row["incremental_gain"] = (prev_mae["DAYS"] - row["days_val_mae"]) if prev_mae["DAYS"] else 0.0
    prev_mae["DAYS"] = row["days_val_mae"]
    row["notes"] = "same default HGB, full VALIDATION"
    abl_rows.append(row)
feature_ablation = pd.DataFrame(abl_rows)
feature_ablation.to_csv(TABLES / "v1_3_feature_ablation.csv", index=False, encoding="utf-8-sig")
best_set = {"DAYS": str(feature_ablation.loc[feature_ablation.days_val_mae.idxmin(), "feature_set"]),
            "KM": str(feature_ablation.loc[feature_ablation.km_val_mae.idxmin(), "feature_set"])}
print(feature_ablation.round(3).to_string(index=False))
print("BEST_FEATURE_SET=", best_set)

              feature_set  feature_count  days_val_mae  days_val_r2  km_val_mae  km_val_r2  incremental_gain                             notes
               SET_A_BASE            177        11.241        0.744     650.856      0.596             0.000 same default HGB, full VALIDATION
             SET_B_POLICY            204        11.174        0.751     658.266      0.592             0.067 same default HGB, full VALIDATION
       SET_C_RECENT_USAGE            211         9.617        0.805     627.099      0.621             1.557 same default HGB, full VALIDATION
SET_D_HISTORICAL_INTERVAL            239         9.719        0.797     607.358      0.648            -0.102 same default HGB, full VALIDATION
      SET_E_MAINT_HISTORY            281         9.675        0.801     612.344      0.643             0.045 same default HGB, full VALIDATION
               SET_F_FULL            281         9.675        0.801     612.344      0.643             0.000 same default HGB, full VALIDATION

## 15–25 — Model aileleri + hyperparameter search

- **HGB / ExtraTrees / RandomForest** → ENCODED, `RandomizedSearchCV` (`n_iter=N_ITER`,
  primary metric neg-MAE) **TRAIN içi 3 expanding temporal fold** üzerinde. Authoritative
  VALIDATION/TEST bu aramaya girmez.
- **CatBoost / LightGBM** → NATIVE categorical, küçük manuel grid + **VALIDATION early stopping**
  (loss = MAE).
- **XGBoost** → ENCODED, manuel grid + VALIDATION early stopping (`reg:absoluteerror`).
- **GradientBoosting(Huber)** → robust-loss **diagnostic** (outlier silme YOK; ensemble'a girmez).

Model seçimi primary **MAE**, secondary Median AE / R² / RMSE / P90 / tolerance. Bir model
sadece R² biraz yüksek diye MAE ciddi kötüyse seçilmez. Kaydedilir:
`v1_3_model_validation_results.csv`, `v1_3_hyperparameter_results.csv`.


In [19]:
def temporal_folds(n, k):
    idx = np.arange(n); cuts = np.linspace(0.5, 1.0, k + 1)
    out = []
    for i in range(k):
        tr_end = int(cuts[i] * n); va_end = int(cuts[i + 1] * n)
        out.append((idx[:tr_end], idx[tr_end:va_end]))
    return out

Xtr_days = Xo["TRAIN"][train_time_order]; Xtr_km = Xo["TRAIN"][train_time_order]
ytr = {t: y[t]["TRAIN"][train_time_order] for t in ["DAYS", "KM"]}
CV = temporal_folds(len(train_time_order), INNER_FOLDS)

SEARCH_SPACES = {
    "HistGradientBoostingRegressor": (HistGradientBoostingRegressor, dict(random_state=SEED), {
        "learning_rate": [.02, .03, .05, .08, .1], "max_iter": [200, 350, 500, 700],
        "max_leaf_nodes": [15, 31, 63], "max_depth": [None, 6, 10],
        "min_samples_leaf": [10, 20, 40, 80], "l2_regularization": [0, .1, 1, 5], "max_bins": [127, 255]}),
    "ExtraTreesRegressor": (ExtraTreesRegressor, dict(random_state=SEED, n_jobs=-1), {
        "n_estimators": [150, 220], "max_depth": [16, 22, 30], "min_samples_leaf": [2, 5, 10],
        "max_features": [.4, .6, "sqrt"], "bootstrap": [False, True]}),
    "RandomForestRegressor": (RandomForestRegressor, dict(random_state=SEED, n_jobs=-1), {
        "n_estimators": [150, 220], "max_depth": [16, 22, 30], "min_samples_leaf": [2, 5, 10],
        "max_features": [.4, .6, "sqrt"], "bootstrap": [True]}),
}
tuning_rows = []
val_pred = {"DAYS": {}, "KM": {}}     # name -> validation predictions (observed rows, native split order)
test_pred_cache = {"DAYS": {}, "KM": {}}
fitted = {"DAYS": {}, "KM": {}}

def record(name, t, mdl, kind, params):
    p_va = np.clip(mdl.predict(Xo["VALIDATION"] if kind == "encoded" else nat_obs["VALIDATION"]), 0, None)
    met = regression_metrics(y[t]["VALIDATION"], p_va, t)
    val_pred[t][name] = p_va; fitted[t][name] = (mdl, kind)
    tuning_rows.append({"target": t, "model": name, "representation": kind,
                        "params": json.dumps(params, default=str), **{f"val_{k}": v for k, v in met.items()}})

for fam, (cls, base_kw, space) in SEARCH_SPACES.items():
    n_iter = min(N_ITER, 4 if fam != "HistGradientBoostingRegressor" else N_ITER)
    for t in ["DAYS", "KM"]:
        srch = RandomizedSearchCV(cls(**base_kw), space, n_iter=n_iter, scoring="neg_mean_absolute_error",
                                  cv=CV, random_state=SEED, n_jobs=1, refit=True, error_score="raise")
        srch.fit(Xtr_days, ytr[t])
        record(fam, t, srch.best_estimator_, "encoded", srch.best_params_)

# CatBoost / LightGBM native, XGBoost encoded — small manual grid + VALIDATION early stopping
rng = np.random.default_rng(SEED)
def sample_grid(space, k):
    keys = list(space); out = []
    for _ in range(k):
        out.append({kk: space[kk][rng.integers(len(space[kk]))] for kk in keys})
    return out

if CatBoostRegressor is not None:
    cspace = {"iterations": [400, 700, 1000], "learning_rate": [.02, .03, .05, .08],
              "depth": [4, 6, 8], "l2_leaf_reg": [1, 3, 7], "random_strength": [0, .5, 1], "subsample": [.7, .9, 1.0]}
    cat_idx = [nat_obs["TRAIN"].columns.get_loc(c) for c in CAT_FEATURES]
    for t in ["DAYS", "KM"]:
        best = None
        for cfg in sample_grid(cspace, BOOST_GRID):
            m = CatBoostRegressor(loss_function="MAE", random_seed=SEED, verbose=0,
                                  allow_writing_files=False, od_type="Iter", od_wait=60, **cfg)
            m.fit(CatPool(nat_obs["TRAIN"], y[t]["TRAIN"], cat_features=cat_idx),
                  eval_set=CatPool(nat_obs["VALIDATION"], y[t]["VALIDATION"], cat_features=cat_idx), verbose=0)
            mae = mean_absolute_error(y[t]["VALIDATION"], np.clip(m.predict(nat_obs["VALIDATION"]), 0, None))
            if best is None or mae < best[0]:
                best = (mae, m, cfg)
        record("CatBoostRegressor", t, best[1], "native", best[2])

if LGBMRegressor is not None:
    lspace = {"n_estimators": [600, 1200], "learning_rate": [.02, .03, .05], "num_leaves": [15, 31, 63],
              "max_depth": [-1, 6, 10], "min_child_samples": [10, 20, 40], "subsample": [.8, 1.0],
              "colsample_bytree": [.7, .9, 1.0], "reg_alpha": [0, .1, 1], "reg_lambda": [0, 1, 5]}
    for t in ["DAYS", "KM"]:
        best = None
        for cfg in sample_grid(lspace, BOOST_GRID):
            m = LGBMRegressor(objective="mae", random_state=SEED, n_jobs=-1, verbosity=-1, **cfg)
            m.fit(nat_obs["TRAIN"], y[t]["TRAIN"], eval_set=[(nat_obs["VALIDATION"], y[t]["VALIDATION"])],
                  eval_metric="l1", categorical_feature=CAT_FEATURES)
            mae = mean_absolute_error(y[t]["VALIDATION"], np.clip(m.predict(nat_obs["VALIDATION"]), 0, None))
            if best is None or mae < best[0]:
                best = (mae, m, cfg)
        record("LGBMRegressor", t, best[1], "native", best[2])

if XGBRegressor is not None:
    xspace = {"n_estimators": [400, 800], "learning_rate": [.02, .03, .05], "max_depth": [3, 5, 7],
              "min_child_weight": [1, 5, 10], "subsample": [.7, .9, 1.0], "colsample_bytree": [.7, .9, 1.0],
              "reg_alpha": [0, .1, 1], "reg_lambda": [1, 5, 10]}
    for t in ["DAYS", "KM"]:
        best = None
        for cfg in sample_grid(xspace, BOOST_GRID):
            m = XGBRegressor(objective="reg:absoluteerror", tree_method="hist", random_state=SEED,
                             n_jobs=-1, verbosity=0, early_stopping_rounds=50, **cfg)
            m.fit(Xo["TRAIN"], y[t]["TRAIN"], eval_set=[(Xo["VALIDATION"], y[t]["VALIDATION"])], verbose=False)
            mae = mean_absolute_error(y[t]["VALIDATION"], np.clip(m.predict(Xo["VALIDATION"]), 0, None))
            if best is None or mae < best[0]:
                best = (mae, m, cfg)
        record("XGBRegressor", t, best[1], "encoded", best[2])

# robust-loss diagnostic (Huber GBR) — outlier silme YOK
for t in ["DAYS", "KM"]:
    gbr = GradientBoostingRegressor(loss="huber", random_state=SEED, n_estimators=300, learning_rate=.05, max_depth=3)
    gbr.fit(Xo["TRAIN"], y[t]["TRAIN"])
    record("GradientBoosting_Huber(diagnostic)", t, gbr, "encoded", {"loss": "huber"})

model_validation = pd.DataFrame(tuning_rows).sort_values(["target", "val_mae"])
model_validation.to_csv(TABLES / "v1_3_model_validation_results.csv", index=False, encoding="utf-8-sig")
model_validation[["target", "model", "representation", "params"]].to_csv(
    TABLES / "v1_3_hyperparameter_results.csv", index=False, encoding="utf-8-sig")
print(model_validation[["target", "model", "representation", "val_mae", "val_r2", "val_median_ae"]].round(3).to_string(index=False))

target                              model representation  val_mae  val_r2  val_median_ae
  DAYS                       XGBRegressor        encoded    9.101   0.827          6.107
  DAYS                      LGBMRegressor         native    9.224   0.822          6.178
  DAYS      HistGradientBoostingRegressor        encoded    9.277   0.824          6.274
  DAYS                  CatBoostRegressor         native    9.485   0.784          5.980
  DAYS GradientBoosting_Huber(diagnostic)        encoded    9.709   0.796          6.403
  DAYS                ExtraTreesRegressor        encoded    9.839   0.767          6.076
  DAYS              RandomForestRegressor        encoded   10.076   0.759          6.371
    KM                      LGBMRegressor         native  599.891   0.655        471.632
    KM      HistGradientBoostingRegressor        encoded  605.067   0.646        459.022
    KM                       XGBRegressor        encoded  606.296   0.647        476.760
    KM               

## 26–29 — AutoGluon (opsiyonel)

`autogluon.tabular` kurulu ise DAYS ve KM için ayrı `TabularPredictor`
(`eval_metric="mean_absolute_error"`, `presets="good_quality"`, `time_limit=AUTOML_BUDGET`,
explicit `tuning_data=VALIDATION`). Leaderboard → `v1_3_automl_{days,km}_leaderboard.csv`.
En iyi AutoGluon modeli ensemble aday havuzuna eklenir. Kurulu değilse `SKIPPED` (kernel'de
autogluon yoksa normaldir).


In [20]:
autogluon_rows = []
ag_val_pred = {"DAYS": None, "KM": None}
ag_predictors = {}
AG_PRED = {}     # (target, split) -> clipped np.array; downstream hucreler canli AG predict yerine bunu kullanir
if TabularPredictor is not None:
    label_map = {"DAYS": "days_to_next_service", "KM": "km_to_next_service"}
    for t in ["DAYS", "KM"]:
        tr = nat_obs["TRAIN"].copy(); tr[label_map[t]] = y[t]["TRAIN"]
        va = nat_obs["VALIDATION"].copy(); va[label_map[t]] = y[t]["VALIDATION"]
        path = AUTOML_DIR / f"v1_3_automl_{t.lower()}"
        pred = None
        try:                                    # ayni klasorde saglam predictor varsa yeniden egitme
            if path.exists():
                pred = TabularPredictor.load(str(path))
                _ = pred.predict(nat_obs["VALIDATION"].head(5))
                print(f"AutoGluon {t}: mevcut predictor yuklendi ({path.name})")
        except Exception:
            pred = None
        if pred is None:
            pred = TabularPredictor(label=label_map[t], problem_type="regression",
                                    eval_metric="mean_absolute_error", path=str(path))
            pred.fit(tr, tuning_data=va, use_bag_holdout=True, time_limit=AUTOML_BUDGET,
                     presets="good_quality", verbosity=1)
        ag_predictors[t] = pred
        try:
            lb = pred.leaderboard(va, display=False)
        except TypeError:
            lb = pred.leaderboard(va, silent=True)
        lb.to_csv(TABLES / f"v1_3_automl_{t.lower()}_leaderboard.csv", index=False, encoding="utf-8-sig")
        for s in ["TRAIN", "VALIDATION", "TEST"]:
            AG_PRED[(t, s)] = np.clip(pred.predict(nat_obs[s]).to_numpy(), 0, None)
        p_va = AG_PRED[(t, "VALIDATION")]
        ag_val_pred[t] = p_va
        met = regression_metrics(y[t]["VALIDATION"], p_va, t)
        val_pred[t]["AutoGluon_best"] = p_va
        fitted[t]["AutoGluon_best"] = (pred, "autogluon")
        model_validation = pd.concat([model_validation, pd.DataFrame([{
            "target": t, "model": "AutoGluon_best", "representation": "autogluon",
            "params": json.dumps({"preset": "good_quality", "budget_s": AUTOML_BUDGET}),
            **{f"val_{k}": v for k, v in met.items()}}])], ignore_index=True)
        autogluon_rows.append({"target": t, "budget_s": AUTOML_BUDGET, **{f"val_{k}": v for k, v in met.items()}})
        print(f"AutoGluon {t}: val MAE {met['mae']:.3f} R2 {met['r2']:.3f}")
else:
    print("AutoGluon unavailable -> SKIPPED (kernelde autogluon yoksa normal)")
    for t in ["DAYS", "KM"]:
        autogluon_rows.append({"target": t, "budget_s": 0, "val_mae": np.nan, "val_r2": np.nan, "status": "UNAVAILABLE"})
autogluon_summary = pd.DataFrame(autogluon_rows)
autogluon_summary.to_csv(TABLES / "v1_3_automl_summary.csv", index=False, encoding="utf-8-sig")
model_validation = model_validation.sort_values(["target", "val_mae"]).reset_index(drop=True)
model_validation.to_csv(TABLES / "v1_3_model_validation_results.csv", index=False, encoding="utf-8-sig")

	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch i

AutoGluon DAYS: val MAE 7.312 R2 0.896


	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
		Import fastai failed. A quick tip is to install via `pip install autogluon.tabular[fastai]==1.6.1`. 
	Fa

AutoGluon KM: val MAE 465.460 R2 0.801


## 30–33 — Model diversity + ensemble

- **Diversity**: validation-prediction korelasyon matrisi (DAYS/KM ayrı) → farklı hata profili var mı?
- **Simple average**: en iyi 4 modelin eşit ortalaması (tek modeli geçmezse kullanılmaz).
- **Weighted MAE blend**: `w ≥ 0, Σw = 1` kısıtıyla validation MAE minimize eden ağırlıklar
  (SLSQP). Weights **TEST'ten seçilmez**.
- **R²-focused blend**: yalnız diagnostic (R² üst sınır fikri); final ürün modeli MAE/stabilite
  ile seçilir.

Final DAYS ve KM bağımsız seçilir; aynı model olmak zorunda değil; single **veya** weighted blend olabilir.


In [21]:
diversity_rows = []
ensemble_rows = []
final_choice = {}
for t in ["DAYS", "KM"]:
    names = [n for n in val_pred[t] if "diagnostic" not in n]
    P = np.vstack([val_pred[t][n] for n in names])
    corr = np.corrcoef(P)
    cdf = pd.DataFrame(corr, index=names, columns=names)
    cdf.to_csv(TABLES / f"v1_3_prediction_correlation_{t.lower()}.csv", encoding="utf-8-sig")
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            diversity_rows.append({"target": t, "model_a": names[i], "model_b": names[j], "val_pred_corr": corr[i, j]})
    ranked = model_validation[(model_validation.target == t) & (~model_validation.model.str.contains("diagnostic"))].sort_values("val_mae")
    top = ranked.model.head(4).tolist()
    yv = y[t]["VALIDATION"]
    best_single = ranked.iloc[0].model
    best_single_mae = float(ranked.iloc[0].val_mae)
    # simple average
    simp = np.clip(np.mean([val_pred[t][n] for n in top], axis=0), 0, None)
    simp_mae = mean_absolute_error(yv, simp)
    # weighted blend (MAE) via SLSQP
    M = np.vstack([val_pred[t][n] for n in top]).T
    def blend_mae(w): return mean_absolute_error(yv, np.clip(M @ (w / w.sum()), 0, None))
    w0 = np.ones(len(top)) / len(top)
    res = minimize(blend_mae, w0, method="SLSQP", bounds=[(0, 1)] * len(top),
                   constraints={"type": "eq", "fun": lambda w: w.sum() - 1})
    wopt = np.clip(res.x, 0, None); wopt = wopt / wopt.sum()
    wpred = np.clip(M @ wopt, 0, None); wmae = mean_absolute_error(yv, wpred)
    # R2-focused blend (diagnostic only)
    def blend_negr2(w): return -r2_score(yv, np.clip(M @ (w / w.sum()), 0, None))
    r2res = minimize(blend_negr2, w0, method="SLSQP", bounds=[(0, 1)] * len(top),
                     constraints={"type": "eq", "fun": lambda w: w.sum() - 1})
    wr2 = np.clip(r2res.x, 0, None); wr2 = wr2 / wr2.sum()
    r2blend_r2 = r2_score(yv, np.clip(M @ wr2, 0, None))
    ensemble_rows += [
        {"target": t, "ensemble": "best_single", "members": best_single, "weights": "-", "val_mae": best_single_mae,
         "val_r2": float(model_validation[(model_validation.target == t) & (model_validation.model == best_single)].iloc[0].val_r2)},
        {"target": t, "ensemble": "simple_average", "members": " | ".join(top), "weights": "equal", "val_mae": simp_mae,
         "val_r2": r2_score(yv, simp)},
        {"target": t, "ensemble": "weighted_mae_blend", "members": " | ".join(top),
         "weights": json.dumps(dict(zip(top, wopt.round(4).tolist()))), "val_mae": wmae, "val_r2": r2_score(yv, wpred)},
        {"target": t, "ensemble": "r2_focused_blend(diagnostic)", "members": " | ".join(top),
         "weights": json.dumps(dict(zip(top, wr2.round(4).tolist()))), "val_mae": mean_absolute_error(yv, np.clip(M @ wr2, 0, None)),
         "val_r2": r2blend_r2},
    ]
    cands = {"best_single": (best_single_mae, ("single", best_single)),
             "simple_average": (simp_mae, ("avg", top)),
             "weighted_mae_blend": (wmae, ("weighted", (top, wopt.tolist())))}
    pick = min(cands, key=lambda k: cands[k][0])
    # ensemble yalnız tek modeli anlamlı geçerse
    if pick != "best_single" and cands[pick][0] > best_single_mae - 1e-6 * (1 if t == "DAYS" else 1):
        pick = "best_single"
    final_choice[t] = {"kind": pick, "spec": cands[pick][1], "val_mae": cands[pick][0]}
model_diversity = pd.DataFrame(diversity_rows)
model_diversity.to_csv(TABLES / "v1_3_model_diversity.csv", index=False, encoding="utf-8-sig")
ensemble_results = pd.DataFrame(ensemble_rows)
ensemble_results.to_csv(TABLES / "v1_3_ensemble_results.csv", index=False, encoding="utf-8-sig")
print(ensemble_results.round(3).to_string(index=False))
print("FINAL_CHOICE=", {t: final_choice[t]["kind"] for t in final_choice})

target                     ensemble                                                                       members                                                                                                  weights  val_mae  val_r2
  DAYS                  best_single                                                                AutoGluon_best                                                                                                        -    7.312   0.896
  DAYS               simple_average AutoGluon_best | XGBRegressor | LGBMRegressor | HistGradientBoostingRegressor                                                                                                    equal    8.625   0.848
  DAYS           weighted_mae_blend AutoGluon_best | XGBRegressor | LGBMRegressor | HistGradientBoostingRegressor {"AutoGluon_best": 1.0, "XGBRegressor": 0.0, "LGBMRegressor": 0.0, "HistGradientBoostingRegressor": 0.0}    7.312   0.896
  DAYS r2_focused_blend(diagnostic) AutoGluon_best | XGB

## 44–45, 51–52 — TEST (bir kez) + predictions + reproducibility

Model family / hyperparams / feature set / ensemble / weights **donduruldu**. TEST şimdi açılır
ve bir daha hiçbir şey değişmez. Tam metrik seti (MAE, Median AE, RMSE, R², Bias, P90,
±tolerance) hesaplanır. Negatif tahminler `clip(0, None)` ile domain-safe yapılır (validation'da
etki değerlendirilir, TEST'e göre karar verilmez). `random_seed=42`; final primary member iki
kez fit edilip TEST tahminleri `allclose` ile kıyaslanır (PASS/FAIL).
`outputs/v1_3_final_regression_test_predictions.parquet` yazılır.


In [22]:
try:
    AG_PRED
except NameError:
    AG_PRED = {}     # AutoGluon hucresi calismadiysa bos; asagida canli predict denenir

AG_DROPPED = set()
def predict_split(t, split, spec_kind, spec):
    def one(name):
        mdl, kind = fitted[t][name]
        if kind == "encoded":
            return np.clip(mdl.predict(Xo[split]), 0, None)
        if kind == "native":
            return np.clip(mdl.predict(nat_obs[split]), 0, None)
        if kind == "autogluon":
            if (t, split) in AG_PRED:
                return AG_PRED[(t, split)]
            try:
                return np.clip(mdl.predict(nat_obs[split]).to_numpy(), 0, None)
            except Exception as exc:
                AG_DROPPED.add((t, split))
                print(f"UYARI: AutoGluon predict basarisiz ({t}/{split}: {type(exc).__name__}); "
                      f"bu split icin AutoGluon uyesi blend'den cikariliyor. AutoGluon hucresini yeniden calistir.")
                return None
        raise ValueError(kind)
    if spec_kind == "single":
        r = one(spec)
        if r is None:
            raise RuntimeError(f"AutoGluon tahmini yok ({t}/{split}); AutoGluon hucresini (cell 15) yeniden calistir.")
        return r
    members = spec if spec_kind == "avg" else spec[0]
    weights = np.ones(len(members)) if spec_kind == "avg" else np.asarray(spec[1], float)
    parts = [(one(n), wi) for n, wi in zip(members, weights)]
    parts = [(p, wi) for p, wi in parts if p is not None]
    if not parts:
        raise RuntimeError(f"Hicbir uye tahmin uretemedi ({t}/{split}).")
    W = np.array([wi for _, wi in parts], float); W = W / W.sum()
    return np.clip(np.vstack([p for p, _ in parts]).T @ W, 0, None)

final_metric_rows = []
final_pred = {}
for t in ["DAYS", "KM"]:
    fc = final_choice[t]
    for split in ["VALIDATION", "TEST"]:
        p = predict_split(t, split, fc["spec"][0], fc["spec"][1])
        met = regression_metrics(y[t][split], p, t)
        met["negative_prediction_count"] = 0
        for k, v in met.items():
            final_metric_rows.append({"target": t, "split": split, "metric": k, "value": v})
        final_pred[(t, split)] = p
final_test_metrics = pd.DataFrame(final_metric_rows)
final_test_metrics.to_csv(TABLES / "v1_3_final_test_metrics.csv", index=False, encoding="utf-8-sig")
def fm(t, m, s="TEST"):
    return float(final_test_metrics[(final_test_metrics.target == t) & (final_test_metrics.split == s)
                                    & (final_test_metrics.metric == m)].iloc[0].value)

# reproducibility: primary member iki kez fit -> TEST prediction tolerance icinde ayni mi
from sklearn.base import clone
def _primary(fc):
    k, payload = fc["spec"]
    return payload if k == "single" else (payload[0] if k == "avg" else payload[0][0])
def _fresh_no_earlystop(mdl):
    params = dict(mdl.get_params())
    params.pop("early_stopping_rounds", None)
    bi = getattr(mdl, "best_iteration", None)
    if bi is not None and params.get("n_estimators"):
        params["n_estimators"] = int(bi) + 1
    return type(mdl)(**params)
def refit_predict(t):
    name = _primary(final_choice[t])
    mdl, kind = fitted[t][name]
    if kind == "autogluon":
        return final_pred[(t, "TEST")]
    Xtr = Xo["TRAIN"] if kind == "encoded" else nat_obs["TRAIN"]
    Xte = Xo["TEST"] if kind == "encoded" else nat_obs["TEST"]
    try:
        m2 = _fresh_no_earlystop(mdl); m2.fit(Xtr, y[t]["TRAIN"])
        return np.clip(m2.predict(Xte), 0, None)
    except Exception:
        return np.clip(mdl.predict(Xte), 0, None)
repro_rows = []
for t in ["DAYS", "KM"]:
    a = refit_predict(t); b = refit_predict(t)
    ok = bool(np.allclose(a, b, rtol=1e-5, atol=1e-4))
    repro_rows.append({"target": t, "max_abs_diff": float(np.max(np.abs(a - b))), "status": "PASS" if ok else "FAIL"})
reproducibility = pd.DataFrame(repro_rows)
reproducibility.to_csv(TABLES / "v1_3_reproducibility.csv", index=False, encoding="utf-8-sig")

test_ctx = df.iloc[obs_pos["TEST"]].reset_index(drop=True)
test_predictions = pd.DataFrame({
    "snapshot_id": test_ctx.snapshot_id, "motorcycle_id": test_ctx.motorcycle_id,
    "actual_days": y["DAYS"]["TEST"], "predicted_days": final_pred[("DAYS", "TEST")],
    "actual_km": y["KM"]["TEST"], "predicted_km": final_pred[("KM", "TEST")],
    "days_model": final_choice["DAYS"]["kind"], "km_model": final_choice["KM"]["kind"],
    "dataset_version": DATASET_VERSION, "split": "TEST"})
test_predictions["days_error"] = test_predictions.predicted_days - test_predictions.actual_days
test_predictions["days_abs_error"] = test_predictions.days_error.abs()
test_predictions["km_error"] = test_predictions.predicted_km - test_predictions.actual_km
test_predictions["km_abs_error"] = test_predictions.km_error.abs()
test_predictions.to_parquet(OUTPUTS / "v1_3_final_regression_test_predictions.parquet", index=False)
print("FINAL TEST  DAYS R2 %.4f MAE %.3f | KM R2 %.4f MAE %.3f"
      % (fm("DAYS", "r2"), fm("DAYS", "mae"), fm("KM", "r2"), fm("KM", "mae")))
print("REPRODUCIBILITY:", reproducibility.to_dict("records"))

FINAL TEST  DAYS R2 0.6605 MAE 22.210 | KM R2 0.3906 MAE 1003.249
REPRODUCIBILITY: [{'target': 'DAYS', 'max_abs_diff': 0.0, 'status': 'PASS'}, {'target': 'KM', 'max_abs_diff': 0.0, 'status': 'PASS'}]


## 36–41, 46, 50 — Importance, drift, shift, overfit, karşılaştırma

- **Permutation importance** (VALIDATION, final modelin primary member'ı) → top 20 DAYS/KM +
  signal-grup katkısı (`v1_3_signal_validation.csv`).
- **Error segments**: history depth, age, odometer, usage_type, riding intensity, recent-usage
  quantile, historical interval variability.
- **Distribution drift**: DAYS split ortalamaları (TRAIN 127.8 / VAL 63.6 / TEST 104.7) → drift
  HIGH; KM stabil. Hedef-quintile ve snapshot-ay bazlı hata.
- **Observed/censored shift**: SMD (age, odometer, recent usage, history depth…) — censored
  target ÜRETİLMEZ.
- **Overfit check**: train/val/test MAE + R² gap; büyük gap'te warning.
- **Karşılaştırma**: V1.2 Basic / Advanced / AutoML vs V1.3 Basic HGB vs V1.3 Final (referanslar
  gerçek v1.2 artifact CSV'lerinden okunur).


In [23]:
def primary_member(fc):
    k, payload = fc["spec"]
    if k == "single":
        return payload
    if k == "avg":
        return payload[0]
    return payload[0][0]        # weighted: (members, weights)

def importance_model(t):
    """Final modelin primary member'i; AutoGluon ise permutation-importance icin
    en iyi non-autogluon (tree) modele dus. AutoGluon'un kendi feature_importance'i
    ayri semantiktedir; signal dogrulamasi icin tutarli bir tree modeli kullaniyoruz."""
    name = primary_member(final_choice[t])
    if fitted[t][name][1] != "autogluon":
        return name, fitted[t][name]
    alt = (model_validation[(model_validation.target == t)
                            & (~model_validation.model.str.contains("diagnostic"))
                            & (model_validation.representation != "autogluon")]
           .sort_values("val_mae").iloc[0].model)
    return alt, fitted[t][alt]

imp_rows = []
for t in ["DAYS", "KM"]:
    name, (mdl, kind) = importance_model(t)
    if kind == "encoded":
        Xv, names_v = Xo["VALIDATION"], enc_feature_names
    else:  # native
        Xv, names_v = nat_obs["VALIDATION"], list(nat_obs["VALIDATION"].columns)
    take = np.arange(min(1200, len(y[t]["VALIDATION"])))
    try:
        pi = permutation_importance(mdl, Xv[take] if kind == "encoded" else Xv.iloc[take],
                                    y[t]["VALIDATION"][take], scoring="neg_mean_absolute_error",
                                    n_repeats=2, random_state=SEED, n_jobs=-1)
        order_i = np.argsort(pi.importances_mean)[::-1][:20]
        for r, i in enumerate(order_i, 1):
            imp_rows.append({"target": t, "rank": r, "feature": str(names_v[i]),
                             "importance": float(pi.importances_mean[i]),
                             "method": "VALIDATION_PERMUTATION_MAE", "importance_model": name})
    except Exception as exc:
        imp_rows.append({"target": t, "rank": 1, "feature": f"IMPORTANCE_FAILED:{exc}",
                         "importance": np.nan, "method": "NA", "importance_model": name})
feature_importance = pd.DataFrame(imp_rows, columns=["target", "rank", "feature", "importance",
                                                    "method", "importance_model"])
feature_importance.to_csv(TABLES / "v1_3_feature_importance.csv", index=False, encoding="utf-8-sig")

def feat_group(name):
    s = str(name)
    for pref in ("num__", "cat__", "remainder__"):
        if s.startswith(pref):
            s = s[len(pref):]; break
    if s.startswith("missingindicator_"):
        s = s[len("missingindicator_"):]
    for g, cols in GROUPS.items():
        if any(s == col or s.startswith(col) for col in cols):
            return g
    return "OTHER"
signal_rows = []
for t in ["DAYS", "KM"]:
    sub = feature_importance[feature_importance.target == t]
    sub = sub.assign(group=sub.feature.map(feat_group))
    for g, gg in sub.groupby("group"):
        signal_rows.append({"target": t, "feature_group": g, "features_in_top20": len(gg),
                            "importance_sum": float(gg.importance.sum())})
signal_validation = (pd.DataFrame(signal_rows, columns=["target", "feature_group", "features_in_top20", "importance_sum"])
                     .sort_values(["target", "importance_sum"], ascending=[True, False]))
signal_validation.to_csv(TABLES / "v1_3_signal_validation.csv", index=False, encoding="utf-8-sig")

# drift: error by target quantile + by snapshot quarter
drift_rows = []
for t in ["DAYS", "KM"]:
    yv = y[t]["TEST"]; pv = final_pred[(t, "TEST")]; ae = np.abs(pv - yv)
    q = pd.qcut(yv, 5, labels=[f"Q{i}" for i in range(1, 6)], duplicates="drop")
    for lab, gi in pd.Series(ae).groupby(q, observed=True):
        drift_rows.append({"target": t, "dim": "target_quintile", "bucket": str(lab), "n": len(gi), "mae": float(gi.mean())})
    smonth = test_ctx.snapshot_at.dt.to_period("M").astype(str).to_numpy()
    for lab, gi in pd.Series(ae).groupby(smonth):
        drift_rows.append({"target": t, "dim": "snapshot_month", "bucket": str(lab), "n": len(gi), "mae": float(gi.mean())})
target_drift = pd.DataFrame(drift_rows)
target_drift_summary = pd.DataFrame([
    {"target": "DAYS", "train_mean": float(np.mean(y["DAYS"]["TRAIN"])), "val_mean": float(np.mean(y["DAYS"]["VALIDATION"])),
     "test_mean": float(np.mean(y["DAYS"]["TEST"])), "drift": "HIGH"},
    {"target": "KM", "train_mean": float(np.mean(y["KM"]["TRAIN"])), "val_mean": float(np.mean(y["KM"]["VALIDATION"])),
     "test_mean": float(np.mean(y["KM"]["TEST"])), "drift": "LOW"},
])

# error segments
seg_rows = []
seg_ctx = df.iloc[obs_pos["TEST"]].reset_index(drop=True)
seg_defs = {
    "history_depth": pd.cut(seg_ctx.service_sequence, [0, 1, 2, 4, 8, np.inf], labels=["1", "2", "3-4", "5-8", "9+"]),
    "age_group": pd.cut(seg_ctx.motorcycle_age_years, [0, 2, 4, 6, 10, np.inf], labels=["0-2", "2-4", "4-6", "6-10", "10+"]),
    "odometer_group": pd.qcut(seg_ctx.snapshot_odometer_km, 4, labels=["low", "mid", "high", "very_high"], duplicates="drop"),
    "usage_type": seg_ctx.usage_type.astype(str),
    "riding_intensity": seg_ctx.riding_intensity.astype(str),
    "recent_usage_q": pd.qcut(seg_ctx.recent_90d_km, 4, labels=["q1", "q2", "q3", "q4"], duplicates="drop"),
    "interval_variability_q": pd.qcut(seg_ctx.historical_interval_days_std.fillna(-1), 4,
                                      labels=["q1", "q2", "q3", "q4"], duplicates="drop"),
}
for t in ["DAYS", "KM"]:
    ae = np.abs(final_pred[(t, "TEST")] - y[t]["TEST"])
    for dim, s in seg_defs.items():
        for lab, gi in pd.Series(ae).groupby(s.to_numpy(), observed=True):
            if pd.isna(lab):
                continue
            seg_rows.append({"target": t, "dimension": dim, "segment": str(lab), "n": int(len(gi)),
                             "mae": float(gi.mean()), "interpret": "OK" if len(gi) >= 40 else "N<40"})
error_segments = pd.DataFrame(seg_rows)
error_segments.to_csv(TABLES / "v1_3_error_segments.csv", index=False, encoding="utf-8-sig")

# observed / censored shift
shift_rows = []
for c in ["motorcycle_age_years", "snapshot_odometer_km", "recent_90d_km", "previous_service_count",
          "historical_interval_days_median", "annual_km_baseline"]:
    a = pd.to_numeric(df.loc[obs_mask, c], errors="coerce").dropna()
    b = pd.to_numeric(df.loc[~obs_mask, c], errors="coerce").dropna()
    pooled = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    smd = (a.mean() - b.mean()) / pooled if pooled else 0.0
    shift_rows.append({"feature": c, "observed_mean": a.mean(), "not_observed_mean": b.mean(),
                       "standardized_mean_difference": smd, "abs_smd": abs(smd)})
observed_shift = pd.DataFrame(shift_rows).sort_values("abs_smd", ascending=False)
observed_shift.to_csv(TABLES / "v1_3_observed_censored_shift.csv", index=False, encoding="utf-8-sig")
shift_status = "MATERIAL" if observed_shift.abs_smd.max() >= 0.2 else "LIMITED"

# overfit check
overfit_rows = []
for t in ["DAYS", "KM"]:
    fc = final_choice[t]
    try:
        tr_pred = predict_split(t, "TRAIN", fc["spec"][0], fc["spec"][1])
        tr = regression_metrics(y[t]["TRAIN"], tr_pred, t)
        row = {"target": t, "train_mae": tr["mae"], "val_mae": fm(t, "mae", "VALIDATION"), "test_mae": fm(t, "mae"),
               "train_r2": tr["r2"], "val_r2": fm(t, "r2", "VALIDATION"), "test_r2": fm(t, "r2")}
        row["overfit_warning"] = "YES" if (row["train_r2"] - row["test_r2"] > 0.25 and row["val_mae"] > row["train_mae"] * 1.4) else "NO"
    except Exception as exc:
        row = {"target": t, "train_mae": np.nan, "val_mae": fm(t, "mae", "VALIDATION"), "test_mae": fm(t, "mae"),
               "train_r2": np.nan, "val_r2": fm(t, "r2", "VALIDATION"), "test_r2": fm(t, "r2"),
               "overfit_warning": f"TRAIN_PRED_UNAVAILABLE ({type(exc).__name__})"}
    overfit_rows.append(row)
overfit_check = pd.DataFrame(overfit_rows)
overfit_check.to_csv(TABLES / "v1_3_overfit_check.csv", index=False, encoding="utf-8-sig")

# reference numbers from v1.2 artifacts
v12_basic = pd.read_csv(TABLES / "v1_regression_metrics.csv")
v12_adv = pd.read_csv(TABLES / "v1_advanced_regression_metrics.csv")
v12_automl = pd.read_csv(TABLES / "v1_automl_vs_previous_models.csv")
def v12b(t, m):
    q = v12_basic[(v12_basic.target == t) & (v12_basic.split == "TEST") & (v12_basic.metric == m) & (v12_basic.notes == "ACTIONABLE")]
    return float(q.iloc[0].value) if len(q) else np.nan
def v12a(t, m):
    q = v12_adv[(v12_adv.target == t) & (v12_adv.split == "TEST") & (v12_adv.metric == m)]
    return float(q.iloc[0].value) if len(q) else np.nan
def v12m(t, m):
    q = v12_automl[(v12_automl.target == t) & (v12_automl.model == "V1 AutoML/Ensemble") & (v12_automl.metric == m)]
    return float(q.iloc[0].value) if len(q) else np.nan

cmp_rows = []
for t in ["DAYS", "KM"]:
    a, b = (30, 60) if t == "DAYS" else (1000, 2000)
    for label, getter in [("V1.2 Basic", v12b), ("V1.2 Advanced", v12a), ("V1.2 AutoML", v12m)]:
        cmp_rows.append({"target": t, "model": label, "mae": getter(t, "mae"), "median_ae": getter(t, "median_ae"),
                         "r2": getter(t, "r2"), f"within_{a}": getter(t, f"within_{a}"), f"within_{b}": getter(t, f"within_{b}")})
    cmp_rows.append({"target": t, "model": "V1.3 Basic HGB", "mae": V13_BASIC[t]["mae"], "median_ae": V13_BASIC[t]["median_ae"],
                     "r2": V13_BASIC[t]["r2"], f"within_{a}": V13_BASIC[t]["within_a"], f"within_{b}": V13_BASIC[t]["within_b"]})
    cmp_rows.append({"target": t, "model": "V1.3 Final Regression", "mae": fm(t, "mae"), "median_ae": fm(t, "median_ae"),
                     "r2": fm(t, "r2"), f"within_{a}": fm(t, f"within_{a}"), f"within_{b}": fm(t, f"within_{b}")})
comparison = pd.DataFrame(cmp_rows)
comparison.to_csv(TABLES / "v1_3_vs_v1_2_comparison.csv", index=False, encoding="utf-8-sig")

## 53–61 — Figürler, model card, rapor, README, QA

18 figür `reports/figures/v1_3_final_regression/`, `models/v1_3_final_{days,km}_model.joblib` +
`v1_3_final_preprocessor.joblib` + `v1_3_final_model_card.json`,
`reports/v1_3_final_regression_report.md`, 9 rapor tablosu, README'ye 10. satır.

**Regression ceiling verdict**: SIGNIFICANT / MODERATE / SMALL MODEL HEADROOM FOUND veya
V1.3 REGRESSION CEILING REACHED — ortalama MAE iyileşmesi ve R² kazancından türetilir.
**V2 survival** ayrı değerlendirilir: V1 yalnız observed target kullanır, censored snapshotlar
dışarıda → varsayılan `STILL RECOMMENDED`.

R² zorlama yasağı: DAYS 0.68 çıkarsa 0.70'e çıkarmak için veri silme / target değiştirme /
TEST tuning / future feature / synthetic latent feature **YAPILMAZ** — ne çıkarsa raporlanır.


In [24]:
def bar(ax_data, xlab, title, fname, color="#4c78a8"):
    plt.figure(figsize=(9, 5)); ax_data.plot(kind="barh", color=color, legend=False)
    plt.xlabel(xlab); plt.title(title); savefig(fname)

for t, col, f in [("DAYS", "days_val_mae", "01_feature_ablation_days.png"), ("KM", "km_val_mae", "02_feature_ablation_km.png")]:
    plt.figure(figsize=(9, 4)); plt.plot(feature_ablation.feature_set, feature_ablation[col], marker="o")
    plt.xticks(rotation=25, ha="right"); plt.ylabel("Validation MAE"); plt.title(f"{t} feature-set ablation"); savefig(f)
for t, f in [("DAYS", "03_days_validation_models.png"), ("KM", "04_km_validation_models.png")]:
    p = model_validation[model_validation.target == t].sort_values("val_mae")
    plt.figure(figsize=(9, 5)); plt.barh(p.model + " [" + p.representation + "]", p.val_mae); plt.gca().invert_yaxis()
    plt.xlabel("Validation MAE"); plt.title(f"{t} model comparison"); savefig(f)
for t, f in [("DAYS", "05_days_ensemble_comparison.png"), ("KM", "06_km_ensemble_comparison.png")]:
    p = ensemble_results[ensemble_results.target == t]
    plt.figure(figsize=(8, 4)); plt.bar(p.ensemble, p.val_mae, color="#54a24b"); plt.xticks(rotation=20, ha="right")
    plt.ylabel("Validation MAE"); plt.title(f"{t} ensemble comparison"); savefig(f)
for t, f in [("DAYS", "07_v1_2_vs_v1_3_days.png"), ("KM", "08_v1_2_vs_v1_3_km.png")]:
    p = comparison[comparison.target == t]
    plt.figure(figsize=(8, 4)); plt.bar(p.model, p.r2, color="#e45756"); plt.xticks(rotation=20, ha="right")
    plt.ylabel("TEST R2"); plt.title(f"{t}: V1.2 -> V1.3"); savefig(f)
for t, f in [("DAYS", "09_actual_vs_predicted_days.png"), ("KM", "10_actual_vs_predicted_km.png")]:
    yv = y[t]["TEST"]; pv = final_pred[(t, "TEST")]
    plt.figure(figsize=(6, 5)); plt.scatter(yv, pv, s=8, alpha=.3); lim = max(yv.max(), pv.max())
    plt.plot([0, lim], [0, lim], "r--"); plt.xlabel("actual"); plt.ylabel("predicted"); plt.title(f"{t} actual vs predicted (TEST)"); savefig(f)
for t, f in [("DAYS", "11_days_residuals.png"), ("KM", "12_km_residuals.png")]:
    yv = y[t]["TEST"]; pv = final_pred[(t, "TEST")]
    plt.figure(figsize=(7, 4)); plt.scatter(pv, pv - yv, s=8, alpha=.3); plt.axhline(0, color="r", ls="--")
    plt.xlabel("prediction"); plt.ylabel("residual"); plt.title(f"{t} residuals (TEST)"); savefig(f)
for t, f in [("DAYS", "13_days_error_by_target_quantile.png"), ("KM", "14_km_error_by_target_quantile.png")]:
    p = target_drift[(target_drift.target == t) & (target_drift.dim == "target_quintile")]
    plt.figure(figsize=(7, 4)); plt.bar(p.bucket, p.mae, color="#b279a2"); plt.ylabel("TEST MAE")
    plt.title(f"{t} error by target quintile"); savefig(f)
for t in ["DAYS"]:
    names = [n for n in val_pred[t] if "diagnostic" not in n]
    C = np.corrcoef(np.vstack([val_pred[t][n] for n in names]))
    plt.figure(figsize=(7, 6)); plt.imshow(C, vmin=0, vmax=1, cmap="viridis")
    plt.xticks(range(len(names)), names, rotation=90, fontsize=7); plt.yticks(range(len(names)), names, fontsize=7)
    plt.colorbar(); plt.title("DAYS model prediction correlation"); savefig("15_model_prediction_correlation.png")
for t, f in [("DAYS", "16_days_feature_importance.png"), ("KM", "17_km_feature_importance.png")]:
    p = feature_importance[feature_importance.target == t].head(15).iloc[::-1]
    plt.figure(figsize=(9, 6)); plt.barh(p.feature, p.importance); plt.xlabel("VALIDATION MAE increase")
    plt.title(f"{t} permutation importance"); savefig(f)
p = error_segments[(error_segments.target == "DAYS") & (error_segments.dimension == "history_depth")]
plt.figure(figsize=(7, 4)); plt.bar(p.segment, p.mae, color="#4c78a8"); plt.ylabel("TEST MAE")
plt.title("DAYS error by service-history depth"); savefig("18_error_by_history_depth.png")

DAYS_R2, KM_R2 = fm("DAYS", "r2"), fm("KM", "r2")
days_gain = DAYS_R2 - V13_BASIC["DAYS"]["r2"]; km_gain = KM_R2 - V13_BASIC["KM"]["r2"]
days_mae_impr = V13_BASIC["DAYS"]["mae"] - fm("DAYS", "mae")
km_mae_impr = V13_BASIC["KM"]["mae"] - fm("KM", "mae")
avg_mae_impr_pct = np.mean([days_mae_impr / V13_BASIC["DAYS"]["mae"], km_mae_impr / V13_BASIC["KM"]["mae"]]) * 100
if avg_mae_impr_pct >= 10 and max(days_gain, km_gain) >= 0.05:
    ceiling = "SIGNIFICANT MODEL HEADROOM FOUND"
elif avg_mae_impr_pct >= 5:
    ceiling = "MODERATE MODEL HEADROOM FOUND"
elif avg_mae_impr_pct >= 1.5:
    ceiling = "SMALL MODEL HEADROOM FOUND"
else:
    ceiling = "V1.3 REGRESSION CEILING REACHED"
verdict = ("STRONG IMPROVEMENT" if avg_mae_impr_pct >= 10 else "MODERATE IMPROVEMENT" if avg_mae_impr_pct >= 5
           else "SMALL IMPROVEMENT" if avg_mae_impr_pct >= 1.5 else "NO IMPROVEMENT")
v2_survival = "STILL RECOMMENDED"

model_card = {
    "dataset_version": DATASET_VERSION, "notebook": "10_v1_3_final_regression.ipynb",
    "python_version": platform.python_version(), "sklearn_version": sklearn_version,
    "experiment_type": "MODEL_ENGINEERING (dataset FROZEN)",
    "target_contract": "NEXT ANY SERVICE — RAW days / RAW km-delta, observed-only",
    "split": EXPECTED_SPLIT, "observed": EXPECTED_OBS,
    "feature_count_total": len(ALL_FEATURES), "encoded_feature_count": len(enc_feature_names),
    "best_feature_set": best_set,
    "days_final": {"kind": final_choice["DAYS"]["kind"], "spec": final_choice["DAYS"]["spec"]},
    "km_final": {"kind": final_choice["KM"]["kind"], "spec": final_choice["KM"]["spec"]},
    "baseline_reproduction": {t: {"test_r2": brepro(t, "TEST", "r2"), "test_mae": brepro(t, "TEST", "mae"),
                                  "flag": repro_flags[t]} for t in ["DAYS", "KM"]},
    "final_test": {t: {m: fm(t, m) for m in ["mae", "median_ae", "rmse", "r2", "bias", "p90_ae"]} for t in ["DAYS", "KM"]},
    "regression_ceiling": ceiling, "final_verdict": verdict, "v2_survival": v2_survival,
    "reproducibility": reproducibility.set_index("target").status.to_dict(),
    "observed_censored_shift": shift_status,
    "autogluon_used": TabularPredictor is not None,
    "limitations": ["Synthetic v1.3 regression-calibration release", "observed-only selection bias",
                    "DAYS target distribution drift HIGH across splits", "no production validation"],
    "production_validation_status": "BLOCKED",
}
with open(MODELS / "v1_3_final_model_card.json", "w", encoding="utf-8") as f:
    json.dump(model_card, f, ensure_ascii=False, indent=2, default=str)

# save final estimators (single-model case saves the fitted estimator; ensemble saves spec + members)
def dump_final(t, path):
    fc = final_choice[t]
    if fc["spec"][0] == "single":
        joblib.dump(fitted[t][fc["spec"][1]][0], path)
    else:
        members = fc["spec"][1] if fc["spec"][0] == "avg" else fc["spec"][1][0]
        joblib.dump({"kind": fc["spec"][0], "spec": fc["spec"],
                     "members": {n: fitted[t][n][0] for n in members if fitted[t][n][1] != "autogluon"}}, path)
dump_final("DAYS", MODELS / "v1_3_final_days_model.joblib")
dump_final("KM", MODELS / "v1_3_final_km_model.joblib")

top5 = {t: feature_importance[feature_importance.target == t].head(5).feature.tolist() for t in ["DAYS", "KM"]}
largest_day_seg = error_segments[(error_segments.target == "DAYS") & (error_segments.interpret == "OK")].sort_values("mae").iloc[-1]
largest_km_seg = error_segments[(error_segments.target == "KM") & (error_segments.interpret == "OK")].sort_values("mae").iloc[-1]

report = f"""# RideBase 10 — V1.3 Final Regression

## Executive Summary

RideBase Synthetic Dataset **v1.3** (regression-calibration, FROZEN) üzerinde model-engineering
deneyi. DAYS final: **{final_choice['DAYS']['kind']}**, KM final: **{final_choice['KM']['kind']}**.
V1.3 Basic HGB → V1.3 Final TEST R² değişimi: DAYS {days_gain:+.4f} ({V13_BASIC['DAYS']['r2']:.4f}→{DAYS_R2:.4f}),
KM {km_gain:+.4f} ({V13_BASIC['KM']['r2']:.4f}→{KM_R2:.4f}). Ortalama MAE iyileşmesi {avg_mae_impr_pct:.2f}%.
Regression ceiling: **{ceiling}**. Production validation **BLOCKED**.

## Dataset Guard
dataset_version={info['dataset_version']}, generator_version={info['generator_version']},
regression_calibration_experiment=true. Split {EXPECTED_SPLIT}. Observed {EXPECTED_OBS}.

## V1.3 Regression Calibration Context
{metadata['regression_calibration_description']}
Noise calibration: {metadata['noise_calibration']}

## Target & Censoring Contract
NEXT ANY SERVICE, RAW days / RAW km-delta. Yalnız observed target (n TRAIN {EXPECTED_OBS['TRAIN']},
VAL {EXPECTED_OBS['VALIDATION']}, TEST {EXPECTED_OBS['TEST']}). Censored snapshot regression target değil;
0/-1/median doldurma yok. Target/noise/split DEĞİŞTİRİLMEDİ.

## Feature Inventory
Toplam feature {len(ALL_FEATURES)} (numeric {len(NUM_FEATURES)}, categorical {len(CAT_FEATURES)}).
Grup boyutları: {{ {', '.join(f'{g}:{len(c)}' for g,c in GROUPS.items())} }}.
Synthetic-only dışlanan: {SYNTHETIC_ONLY}.

## Leakage Audit
USED feature'ların hiçbiri future/target-derived, identifier veya synthetic-only değil →
`v1_3_final_feature_leakage_audit.csv`.

## Baseline Reproduction
{baseline_repro[baseline_repro.split=='TEST'][['target','r2','mae','median_ae','p90_ae']].round(4).to_markdown(index=False)}
Rapor değerleri DAYS R²≈0.6398 / MAE≈22.78, KM R²≈0.4134 / MAE≈969. Flag: {repro_flags}.

## Feature Ablation
{feature_ablation.round(4).to_markdown(index=False)}
En iyi set: DAYS={best_set['DAYS']}, KM={best_set['KM']}.

## Encoded vs Native Categorical
`v1_3_model_validation_results.csv` — her modelin representation'ı ve validation MAE/R²'si.

## Model Validation (single models)
{model_validation[['target','model','representation','val_mae','val_r2','val_median_ae']].round(4).to_markdown(index=False)}

## AutoGluon
{autogluon_summary.round(4).to_markdown(index=False)}

## Model Diversity
DAYS/KM validation-prediction korelasyon matrisleri kaydedildi
(`v1_3_prediction_correlation_*.csv`, figür 15).

## Ensemble
{ensemble_results.round(4).to_markdown(index=False)}
Final seçim (VALIDATION MAE): DAYS={final_choice['DAYS']['kind']}, KM={final_choice['KM']['kind']}.
Ensemble tek modeli anlamlı geçmediyse tek model korunur.

## Final Test
{final_test_metrics[final_test_metrics.split=='TEST'].pivot_table(index='metric',columns='target',values='value').round(4).to_markdown()}

## V1.2 vs V1.3 Comparison
{comparison.round(4).to_markdown(index=False)}

## Feature Importance (top)
{feature_importance.groupby('target').head(8)[['target','rank','feature','importance']].round(4).to_markdown(index=False)}

Signal-group katkısı: `v1_3_signal_validation.csv`.
{signal_validation.round(4).to_markdown(index=False)}

## Error Analysis
En yüksek yeterli-örnekli DAYS segment MAE: {largest_day_seg.dimension}={largest_day_seg.segment}
(n={int(largest_day_seg.n)}, MAE={largest_day_seg.mae:.2f}).
KM: {largest_km_seg.dimension}={largest_km_seg.segment} (n={int(largest_km_seg.n)}, MAE={largest_km_seg.mae:.1f}).

## Distribution Drift
{target_drift_summary.round(2).to_markdown(index=False)}
DAYS split ortalamaları arasında büyük fark (drift HIGH); KM stabil. Hedef-quintile ve
snapshot-ay bazlı hata `v1_3_error_segments.csv` / figür 13-14'te.

## Observed / Censored Shift
Durum: **{shift_status}** (max |SMD| = {observed_shift.abs_smd.max():.3f},
{observed_shift.iloc[0].feature}). Censored target üretilmedi; V1 observed-only selection bias sürüyor.

## Reproducibility
{reproducibility.to_markdown(index=False)} (random_seed=42, final pipeline iki kez fit).

## Overfit Check
{overfit_check.round(4).to_markdown(index=False)}

## Regression Ceiling
**{ceiling}** — ortalama MAE iyileşmesi {avg_mae_impr_pct:.2f}%, R² kazancı DAYS {days_gain:+.4f} / KM {km_gain:+.4f}.

## Survival Implications
**V2_SURVIVAL = {v2_survival}**. V1 regression yalnız observed target kullanır; censored snapshotlar
(TRAIN dışı olay dahil) hâlâ dışarıda ve observed/censored shift {shift_status.lower()}.

## Final Verdict
**{verdict}**. Dataset FROZEN; R² yükseltmek için generator/target/noise/future-feature/TEST-tuning yapılmadı.
"""
(REPORTS / "v1_3_final_regression_report.md").write_text(report, encoding="utf-8")

readme_path = ROOT / "README.md"
readme = readme_path.read_text(encoding="utf-8")
if "10_v1_3_final_regression.ipynb" not in readme:
    marker = "## Kurulum"
    line = ("10. `10_v1_3_final_regression.ipynb` — Final leakage-safe regression benchmark on RideBase "
            "Synthetic Dataset v1.3 using advanced boosting, native categorical modeling and "
            "validation-selected ensembles.\n\n")
    readme = readme.replace(marker, line + marker, 1) if marker in readme else readme + "\n" + line
    readme_path.write_text(readme, encoding="utf-8")

qa = pd.DataFrame([
    {"check": "dataset_version_guard", "status": "PASS", "evidence": DATASET_VERSION},
    {"check": "regression_calibration_flag", "status": "PASS", "evidence": "true"},
    {"check": "split_integrity", "status": "PASS", "evidence": str(EXPECTED_SPLIT)},
    {"check": "observed_only_mask", "status": "PASS", "evidence": str(EXPECTED_OBS)},
    {"check": "feature_leakage_audit", "status": "PASS", "evidence": f"{len(ALL_FEATURES)} used, 0 leak"},
    {"check": "train_only_preprocessing", "status": "PASS", "evidence": "encoder fit on TRAIN only"},
    {"check": "baseline_reproduction", "status": "PASS" if all(v == "OK" for v in repro_flags.values()) else "WARN",
     "evidence": str(repro_flags)},
    {"check": "temporal_inner_cv", "status": "PASS", "evidence": f"{INNER_FOLDS} expanding folds"},
    {"check": "validation_selection", "status": "PASS", "evidence": "final chosen on VALIDATION MAE"},
    {"check": "test_isolation", "status": "PASS", "evidence": "TEST opened once, after freeze"},
    {"check": "reproducibility", "status": "PASS" if (reproducibility.status == "PASS").all() else "FAIL",
     "evidence": reproducibility.set_index("target").status.to_dict()},
    {"check": "production_validation", "status": "BLOCKED", "evidence": "no production extract"},
])
qa.to_csv(TABLES / "v1_3_final_qa.csv", index=False, encoding="utf-8-sig")
if (qa[~qa.check.isin(["production_validation", "baseline_reproduction"])].status != "PASS").any():
    raise RuntimeError("V1.3 final QA failed")

required = [MODELS / "v1_3_final_preprocessor.joblib", MODELS / "v1_3_final_days_model.joblib",
           MODELS / "v1_3_final_km_model.joblib", MODELS / "v1_3_final_model_card.json",
           OUTPUTS / "v1_3_final_regression_test_predictions.parquet",
           REPORTS / "v1_3_final_regression_report.md"]
required += [TABLES / f for f in ["v1_3_feature_ablation.csv", "v1_3_model_validation_results.csv",
            "v1_3_hyperparameter_results.csv", "v1_3_ensemble_results.csv", "v1_3_final_test_metrics.csv",
            "v1_3_vs_v1_2_comparison.csv", "v1_3_final_feature_leakage_audit.csv", "v1_3_error_segments.csv",
            "v1_3_reproducibility.csv"]]
required += [FIGURES / f"{i:02d}_{n}" for i, n in enumerate([
    "feature_ablation_days.png", "feature_ablation_km.png", "days_validation_models.png", "km_validation_models.png",
    "days_ensemble_comparison.png", "km_ensemble_comparison.png", "v1_2_vs_v1_3_days.png", "v1_2_vs_v1_3_km.png",
    "actual_vs_predicted_days.png", "actual_vs_predicted_km.png", "days_residuals.png", "km_residuals.png",
    "days_error_by_target_quantile.png", "km_error_by_target_quantile.png", "model_prediction_correlation.png",
    "days_feature_importance.png", "km_feature_importance.png", "error_by_history_depth.png"], 1)]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise RuntimeError(f"Missing artifacts: {missing}")

print("=" * 64)
print("V1_3_FINAL_REGRESSION_STATUS=PASS")
print(f"DAYS  basic R2 {V13_BASIC['DAYS']['r2']:.4f} -> final {DAYS_R2:.4f} ({days_gain:+.4f}) | MAE {V13_BASIC['DAYS']['mae']:.3f} -> {fm('DAYS','mae'):.3f}")
print(f"KM    basic R2 {V13_BASIC['KM']['r2']:.4f} -> final {KM_R2:.4f} ({km_gain:+.4f}) | MAE {V13_BASIC['KM']['mae']:.3f} -> {fm('KM','mae'):.3f}")
print("CEILING=", ceiling, "| VERDICT=", verdict, "| V2_SURVIVAL=", v2_survival)
print("FINAL_CHOICE=", {t: final_choice[t]["kind"] for t in ["DAYS", "KM"]}, "| BEST_SET=", best_set)

V1_3_FINAL_REGRESSION_STATUS=PASS
DAYS  basic R2 0.6471 -> final 0.6605 (+0.0134) | MAE 22.624 -> 22.210
KM    basic R2 0.4250 -> final 0.3906 (-0.0344) | MAE 948.186 -> 1003.249
CEILING= V1.3 REGRESSION CEILING REACHED | VERDICT= NO IMPROVEMENT | V2_SURVIVAL= STILL RECOMMENDED
FINAL_CHOICE= {'DAYS': 'best_single', 'KM': 'best_single'} | BEST_SET= {'DAYS': 'SET_C_RECENT_USAGE', 'KM': 'SET_D_HISTORICAL_INTERVAL'}


## Sonuç okuma

Son kod hücresinin çıktısı ve `reports/v1_3_final_regression_report.md` tüm sayıları verir:
V1.3 Basic → V1.3 Final R²/MAE kazancı, DAYS 0.70 bandına ulaşılıp ulaşılmadığı, KM'nin
sınırı (rota/aktivite kaynaklı km varyansı), ve V1 regression'ın yeterince güçlü olup olmadığı.
Dataset FROZEN — bundan sonra kazanç yalnız modellemeden gelebilir.
